In [1]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5" # Old model, supporting prefill

In [16]:
# Helper functions
def add_user_message(messages: list[dict], text: str):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list[dict], text: str):
    messages.append({"role": "assistant", "content": text})    


def chat(messages: list[dict], 
         system_prompt:str|None=None, 
         temperature:float=1.0,
         stop_sequences:list[str]|None=None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system_prompt:
        params["system"] = system_prompt

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        params["thinking"] = {"type": "disabled"} # Thinking is not supported when using assistant message prefill

    response = client.messages.create(**params)
    # print(response)
    return response.content[0].text   

In [14]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages =[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [17]:
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that takes an S3 bucket ARN and extracts just the bucket name from it. The function should handle ARNs in the format 'arn:aws:s3:::bucket-name' or 'arn:aws:s3:::bucket-name/key'."},
 {'task': "Create a JSON object representing an AWS Lambda function's environment variables configuration that includes: database endpoint 'prod-db.us-east-1.rds.amazonaws.com', connection timeout of 30 seconds, and log level set to 'INFO'."},
 {'task': 'Write a regular expression that validates AWS IAM role names according to AWS naming rules: 1-64 characters long, can contain alphanumeric characters, underscores, plus signs, equals signs, commas, periods, at signs, and hyphens.'}]

In [25]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [20]:
from pydantic import BaseModel

class TestUseCase(BaseModel):
    task: str

In [ ]:
def run_prompt(test_case: TestUseCase) -> str:
    """Merges the prompt with the test case input, then returns the result"""
    prompt = f"""
    Please solve the following test:

    {test_case.task}
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)
    

In [44]:
def grade_by_model(test_case: TestUseCase, output: str) -> dict:
    """Grades the response using the model. Returns a grade of "good" or "bad"."""
    prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution

    Original task:
    <task>
    {test_case.task}
    </task>

    Solution to evaluate:
    <solution>
    {output}
    </solution>

    # Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific format:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your evaluation
    - "score": A number between 1-10

    Respond only with JSON. Keep your response concise and direct.
    Examples response shape:
    {{
        "strengths": [],
        "weaknesses": [],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [43]:
grade_by_model(
    TestUseCase(task="Write a Python function that takes a list of numbers and returns the sum of the even numbers."), 
    "```python\ndef sum_even_numbers(numbers):\n    return sum(num for num in numbers if num % 2 == 0)\n```")

{'strengths': ['Concise and Pythonic implementation using generator expression',
  'Correct logic for filtering even numbers using modulo operator',
  'Efficient single-pass solution with O(n) time complexity'],
 'weaknesses': ['No input validation - fails with None or non-iterable inputs',
  'No type hints for better code documentation and IDE support',
  'No handling of non-numeric elements in the list (raises TypeError)'],
 'reasoning': "The solution correctly implements the core requirement using idiomatic Python. However, it lacks robustness for production use - missing input validation, type hints, and error handling. For a simple coding exercise it's good, but production code would need defensive programming practices.",
 'score': 7}

In [45]:
def run_test_case(test_case: TestUseCase):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score: float = model_grade["score"]
    reasoning: str = model_grade["reasoning"]
    

    return {
        "output": output,
        "score": score,
        "test_case": test_case.model_dump(),
        "reasoning": reasoning,
    }

In [46]:
def run_eval(dataset: list):
    """Loads the dataset and calls run_test_case for each one"""
    results = []
    for test_case in dataset:
        result = run_test_case(TestUseCase(**test_case))
        results.append(result)
    
    return results

In [47]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)


results = run_eval(dataset)

In [51]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Solution: Extract S3 Bucket Name from ARN\n\n```python\ndef extract_bucket_name(arn):\n    \"\"\"\n    Extracts the bucket name from an S3 bucket ARN.\n    \n    Args:\n        arn (str): S3 bucket ARN in format 'arn:aws:s3:::bucket-name' \n                   or 'arn:aws:s3:::bucket-name/key'\n    \n    Returns:\n        str: The bucket name extracted from the ARN\n    \n    Examples:\n        >>> extract_bucket_name('arn:aws:s3:::my-bucket')\n        'my-bucket'\n        >>> extract_bucket_name('arn:aws:s3:::my-bucket/path/to/key')\n        'my-bucket'\n    \"\"\"\n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # The bucket name (and potentially key) is after the third colon\n    # Format: arn:aws:s3:::bucket-name or arn:aws:s3:::bucket-name/key\n    if len(parts) >= 6:\n        bucket_and_key = parts[5]\n        \n        # If there's a key (indicated by '/'), extract just the bucket name\n        if '/' in bucket_and_key:\n            buc

In [52]:

from statistics import mean

average_score = mean(result["score"] for result in results)
print(f"Average score: {average_score}")

Average score: 7.333333333333333
